# Week 2 — Wednesday: Visualization with Plotly Express

**DATA 202 · Calvin University**

*(First ~20 min: devotions, announcements, retrieval quiz on Week 1 content — 65 min class, ~45 min of content below.)*

**Today's outline:**

- Part 1 — Why Visualize? (Datasaurus Dozen, variable types)
- Part 2 — Mapping Variables to Visual Encodings (incl. Simpson's Paradox)
- Part 3 — Customizing Charts: Reference + Quick Activity

Watch for two stop-and-check cues along the way: **🎯 Predict First** (guess before we run the code) and **🙋 Quick Check** (a quick verbal question — no code).

---

A DataFrame full of numbers tells you very little by itself. Visualization translates data into **visual metaphors** — distance, position, color, size — that our eyes can process instantly.

Today's goal: learn to **map** variables to visual properties using Plotly Express, and see how to customize a chart's look once the mapping is right.</cell id="cell-0">

---
## Part 1: Why Visualize? · ~15 min

### Summary statistics can lie

Consider the Datasaurus Dozen — a set of very different datasets that share nearly identical summary statistics.</cell id="cell-1">

In [ ]:
import pandas as pd
import plotly.express as px

datasaurus = pd.read_csv("https://cs.calvin.edu/courses/data/202/fsantos/datasets/datasaurus.csv")

sample = datasaurus[datasaurus["dataset"].isin(["away", "bullseye", "dino", "star", "dots"])]
sample.groupby("dataset")[["x", "y"]].mean().round(2)

🎯 **Predict First:** these five datasets have nearly identical `x`/`y` means. Before you run the next cell — do you expect them to *look* similar when plotted, or different? Take a guess.

In [ ]:
# Same means — but look at the actual shapes
px.scatter(sample, x="x", y="y", facet_col="dataset", facet_col_wrap=5,
           width=950, height=280, title="Same statistics, completely different data")

**Takeaway:** always plot your data before trusting any summary. In ML, this matters even more — a model trained blindly on any of these datasets would behave very differently.

### Our dataset: product sales

Simulated sales data for 50 products across categories, seasons, and suppliers.

In [ ]:
sales = pd.read_csv("https://cs.calvin.edu/courses/data/202/fsantos/datasets/product_sales.csv")
sales.head()

### Variable types matter for choosing encodings

Before plotting, classify each variable:

| Type | Sub-type | Example | Good encodings |
|---|---|---|---|
| **Numerical** | Continuous | Sales amount | x/y axis, color gradient, size |
| **Numerical** | Discrete | Units sold | x/y axis, size |
| **Categorical** | Unordered | Product category | color (distinct), symbol, facet |
| **Categorical** | Ordered | Season (Spring→Summer→Fall→Winter) | color (sequential), x-axis order |

🙋 **Quick Check:** Not every column fits neatly into one type. Take `Sales` — it's counted in whole dollars, so technically it's numerical *discrete*. But its range is so wide that in practice it behaves just like a *continuous* variable. Look back at the other columns in `sales` — which ones sit in a similar gray area between two types? What would push you to treat a gray-area column one way versus the other?

---
### 🔨 Task 1 — Classify variables (~5 min)

For each column in `sales`, identify its type (numerical continuous / numerical discrete / categorical unordered / categorical ordered):

| Column | Type | Notes |
|---|---|---|
| Sales | | |
| Returns | | |
| Units Sold | | |
| Profit | | |
| Advertising Spend | | |
| Category | | |
| Season | | |
| Supplier | | |

Which column(s) would you put on the x-axis if you wanted to predict Profit? Why?

*(Answer in your own words here — double-click to edit)*

---
## Part 2: Mapping Variables to Visual Encodings · ~18 min

### The core idea

Plotly Express works by **mapping** DataFrame columns to visual properties:

```python
px.scatter(df, x="col_a", y="col_b", color="col_c", size="col_d", ...)
```

Each argument name is a **visual channel**. Each value is a **column name**. The data drives the visual.</cell id="cell-12">

In [ ]:
# Start simple: two numerical variables
px.scatter(sales, x="Advertising Spend", y="Profit",
           title="Advertising Spend vs. Profit")

In [ ]:
# Add a third variable via color (categorical → distinct hues)
px.scatter(sales, x="Advertising Spend", y="Profit",
           color="Category",
           title="Advertising Spend vs. Profit by Category")

In [ ]:
# Add a fourth variable via size (numerical → point area)
px.scatter(sales, x="Advertising Spend", y="Profit",
           color="Category", size="Units Sold",
           title="Advertising Spend vs. Profit (size = Units Sold)")

In [ ]:
# Facets: split into small multiples by a categorical variable
px.scatter(sales, x="Advertising Spend", y="Profit",
           color="Category", size="Units Sold",
           facet_col="Season",
           title="Advertising Spend vs. Profit by Season")

---
### ⚠️ Wrong mapping, wrong story

Classifying a variable isn't just a labeling exercise — get it wrong and the encoding can scramble the story. `Season` is an *ordered* categorical (Spring → Summer → Fall → Winter), but Plotly has no way to know that unless we tell it. Watch what happens when we just hand it the column:

In [ ]:
# Season treated as an unordered category (the default)
px.box(sales, x="Season", y="Profit", title="Profit by Season (default order)")

Plotly fell back to **alphabetical order** (Fall, Spring, Summer, Winter) because it only sees `Season` as a set of text labels — it doesn't know Spring comes first. Any real trend across the seasons is now scrambled by plotting order, not by the data itself.

Fix it by telling Plotly the real order with `category_orders`:

In [ ]:
# Season treated as the ordered category it actually is
px.box(sales, x="Season", y="Profit",
       category_orders={"Season": ["Spring", "Summer", "Fall", "Winter"]},
       title="Profit by Season (chronological order)")

**Takeaway:** the type you assign a column isn't just bookkeeping — it changes what Plotly does with it by default. An ordered categorical treated as unordered can scramble a trend (like above); the same risk runs the other way too — a high-cardinality or effectively-continuous column (like `Sales`, from the Quick Check above) forced into `color` or `symbol` can blow up your legend into dozens of near-identical hues instead of showing a smooth gradient. Always ask: does the encoding I picked match how the variable actually behaves?

### Simpson's Paradox — when grouping reveals the truth

Look at this overall trend:

In [ ]:
px.scatter(sales, x="Advertising Spend", y="Profit",
           trendline="ols", title="Overall trend")

In [ ]:
# Now break it out by Category — does the trend hold within each group?
px.scatter(sales, x="Advertising Spend", y="Profit",
           color="Category", facet_col="Category", facet_col_wrap=3,
           trendline="ols", title="Trend within each Category")

**Simpson's Paradox:** a trend visible in the whole dataset can reverse — or disappear — within subgroups. Adding `color` or `facet_col` is often what reveals it.

In ML: this is why we always check whether a model's performance holds across subgroups, not just overall.

---
### 🔨 Task 2 — Explore encodings (~5 min)

Create a single scatter plot of `Sales` vs. `Returns` that encodes **at least three additional variables** beyond x and y — use any combination of `color`, `size`, `symbol`, `facet_col`, `facet_row`, or `text`.

Then answer: which of your encodings is most useful? Which feels cluttered or misleading? Why?

In [ ]:
# Your code here


*(Reflection — double-click to edit)*

---
## Part 3: Customizing Charts (Reference) · ~12 min

Once you've picked the right encodings, Plotly Express also gives you full control over how a chart *looks* — titles, axis labels, tick marks, colors, themes. You don't need to memorize this syntax; the cell below is a reference you can come back to whenever you need it.</cell id="fc74e9b0">

In [ ]:
# Reference: common chart customizations — a lookup, not something to memorize
fig = px.scatter(
    sales, x="Advertising Spend", y="Profit", color="Category",
    title="Profit vs. Advertising Spend by Category",           # chart title
    labels={"Advertising Spend": "Advertising Spend ($)",        # axis labels
            "Profit": "Profit ($)",
            "Category": "Product Category"},
    color_discrete_sequence=px.colors.qualitative.Safe,          # color-blind-safe palette
    template="simple_white",                                     # clean theme
)
fig.update_xaxes(dtick=500, tickprefix="$")   # tick spacing + $ prefix
fig.update_yaxes(tickformat=",.0f")           # comma-formatted numbers
fig.show()

**Quick lookup:**

| What | How |
|---|---|
| Title | `title="..."` |
| Axis labels | `labels={"col_name": "Human-readable name"}` |
| Tick spacing | `fig.update_xaxes(dtick=...)` / `fig.update_yaxes(dtick=...)` |
| Tick formatting | `fig.update_xaxes(tickprefix="$", tickformat=",.0f")` |
| Discrete colors | `color_discrete_sequence=px.colors.qualitative.Safe` |
| Theme | `template="simple_white"` (also: `"plotly"`, `"ggplot2"`, `"seaborn"`, `"plotly_dark"`) |

---
### 🔨 Task 3 — Polish a plot (~7 min)

Take this bare-bones chart and improve it using at least **three** of the customizations from the reference above:

```python
px.scatter(sales, x="Units Sold", y="Revenue", color="Season")
```

Ideas: a descriptive title, human-readable axis labels, custom tick spacing/formatting, a color-blind-friendly palette, a clean theme.

**Discuss:** which change made the biggest difference to readability?

In [ ]:
# Your code here
# (Note: "Revenue" does not exist in this dataset — use a column that does)


---
## Reference: Visual Encoding Cheatsheet

| px argument | Variable type it works best with | Notes |
|---|---|---|
| `x`, `y` | Numerical (continuous or discrete) | The primary axes |
| `color` | Categorical (unordered) or Numerical | Distinct hues for categories; gradient for numbers |
| `size` | Numerical (positive) | Area encodes magnitude — use carefully |
| `symbol` | Categorical (few levels, ≤6) | Redundant with color for accessibility |
| `text` | Any (short strings) | Labels on points — gets crowded fast |
| `facet_col` / `facet_row` | Categorical | Small multiples — great for comparisons |
| `animation_frame` | Categorical or ordered | Animated over time or groups |

**Coming up — Friday:** Practice 1 + Quiz 1 (covers SLOs 02A, 02B, 02C)